# GitHub Copilot SDK tracing with Openlayer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/openlayer-ai/openlayer-python/blob/main/examples/tracing/copilot_sdk/copilot_sdk_tracing.ipynb)

This notebook shows how to stream traces from agents built on the [GitHub Copilot SDK](https://github.com/github/copilot-sdk) to Openlayer.

Each `send()` becomes one Openlayer trace:

```
AGENT  "GitHub Copilot"          <- the user prompt in, the final answer out
├─ CHAT_COMPLETION "turn 0"      <- model, tokens, cost, latency
├─ TOOL  "bash"                  <- arguments in, result out
├─ AGENT "subagent: Explore"     <- a `task` dispatch
│   ├─ CHAT_COMPLETION "turn 0"
│   └─ TOOL "view"
└─ CHAT_COMPLETION "turn 1"
```

> **Requires Python 3.11+** — that is `github-copilot-sdk`'s own floor.

## 1. Install

In [ ]:
%pip install openlayer 'github-copilot-sdk>=1.0.11'

## 2. Credentials

The Copilot SDK authenticates with your logged-in GitHub user by default; set `GITHUB_TOKEN` if you'd rather be explicit. You need a GitHub account with Copilot access.

In [ ]:
import os

os.environ["OPENLAYER_API_KEY"] = "YOUR_OPENLAYER_API_KEY_HERE"
os.environ["OPENLAYER_INFERENCE_PIPELINE_ID"] = "YOUR_OPENLAYER_INFERENCE_PIPELINE_ID_HERE"
# os.environ["GITHUB_TOKEN"] = "YOUR_GITHUB_TOKEN_HERE"

## 3. Enable tracing

`init()` is the canonical entry point. With `auto_instrument` on (the default) it
detects every supported SDK you have installed — including the Copilot SDK — and
patches it, so every session you create is traced with no change to the code that
builds them.

If you'd rather be explicit about which sessions are traced, pass
`on_event=openlayer_event_handler()` to `create_session()` instead. Mixing the two
is safe — the patch defers to a handler you supply rather than adding a second one.

In [ ]:
from openlayer.lib import init

init()


## 4. A workspace to work in

Copilot is a coding agent, so give it some real files to look at.

In [ ]:
import pathlib
import tempfile

workspace = pathlib.Path(tempfile.mkdtemp(prefix="openlayer-copilot-"))
(workspace / "app.py").write_text("def greet(name):\n    return f'hi {name}'\n")
(workspace / "README.md").write_text("# Demo\n\nA tiny example project.\n")
print(workspace)  # noqa: T201


## Scenario 1 — a basic session

`PermissionHandler.approve_all` auto-approves tool use. In production you'd supply your own policy; the decision is captured on the tool step either way.

In [ ]:

from copilot import CopilotClient, PermissionHandler


async def basic_session():
    client = CopilotClient(working_directory=str(workspace), log_level="error")
    await client.start()
    try:
        session = await client.create_session(
            working_directory=str(workspace),
            on_permission_request=PermissionHandler.approve_all,
        )
        reply = await session.send_and_wait(
            "List the files here with bash, then summarize the project in one sentence.",
            timeout=300,
        )
        print(reply.data.content)  # noqa: T201
        await session.disconnect()
    finally:
        await client.stop()


await basic_session()

## Scenario 2 — a client-side tool and a subagent

Tools you define with `define_tool` run in *your* process and still appear as `TOOL` steps. A `task` dispatch becomes a nested `AGENT` step, with the subagent's own turns and tool calls inside it.

This also shows `openlayer_event_handler()` — the explicit alternative to `trace_copilot()`, useful when you build sessions yourself. It composes with your own `on_event`: pass both and each still receives every event.

In [ ]:
from copilot import CopilotClient, PermissionHandler, define_tool
from pydantic import BaseModel


class WeatherParams(BaseModel):
    city: str


@define_tool(description="Get the current weather for a city.")
def get_weather(params: WeatherParams) -> str:
    return f"It is 22C and sunny in {params.city}."


async def tools_and_subagent():
    client = CopilotClient(working_directory=str(workspace), log_level="error")
    await client.start()
    try:
        session = await client.create_session(
            working_directory=str(workspace),
            on_permission_request=PermissionHandler.approve_all,
            tools=[get_weather],
        )
        reply = await session.send_and_wait(
            "Do two things: call get_weather for Lisbon, and delegate to a subagent "
            "to read app.py and summarize it.",
            timeout=300,
        )
        print(reply.data.content)  # noqa: T201
        await session.disconnect()
    finally:
        await client.stop()


await tools_and_subagent()

## Scenario 3 — multi-turn session grouping

Each `send()` is its own trace, but they all carry the same Copilot session id, so Openlayer groups them into one session.

In [ ]:
from copilot import CopilotClient, PermissionHandler


async def multi_turn():
    client = CopilotClient(working_directory=str(workspace), log_level="error")
    await client.start()
    try:
        session = await client.create_session(
            working_directory=str(workspace),
            on_permission_request=PermissionHandler.approve_all,
        )
        for prompt in (
            "What files are in this directory?",
            "What does app.py define?",
        ):
            reply = await session.send_and_wait(prompt, timeout=300)
            print(f"Q: {prompt}\nA: {reply.data.content}\n")  # noqa: T201
        await session.disconnect()
    finally:
        await client.stop()


await multi_turn()

## What lands in Openlayer

| | |
|---|---|
| **Row prompt / output** | the user's message and the final assistant answer |
| **Cost** | priced by Openlayer from the real provider and model |
| **Tokens** | input / output / cached / cache-creation, as a non-overlapping partition |
| **Session** | the Copilot session id, so multi-turn conversations group |
| **Tools** | arguments, results, success/failure, permission decisions, MCP server |
| **Subagents** | nested agent steps with their own turns, tools and token totals |

One thing worth knowing: Copilot's own `cost` field is **premium-request units, not dollars** (a flat per-model multiplier, identical on every call regardless of size). Openlayer therefore prices the call itself from the provider and model, and keeps Copilot's figure in step metadata as `copilot_premium_requests`.